<a href="https://colab.research.google.com/github/youngbean0519/Self-Project/blob/main/notebooks/12_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive


# 1. 구글 드라이브 연결
drive.mount("/content/drive")


# 2. 프로젝트 다운로드
PROJECT_DIR = Path("/content/Self-Project")
SRC_DIR = PROJECT_DIR / "src"

if not PROJECT_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/youngbean0519/Self-Project.git",
            str(PROJECT_DIR),
        ],
        check=True,
    )


# 3. 필요한 패키지 설치
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--only-binary=tokenizers",
        "transformers==4.57.6",
        "tokenizers>=0.22.0,<=0.23.0",
        "sentencepiece",
        "kiwipiepy",
        "joblib",
    ],
    check=True,
)


# 4. KoBERT 토크나이저 설치
kobert_package = (
    "git+https://github.com/SKTBrain/KoBERT.git"
    "#egg=kobert_tokenizer&subdirectory=kobert_hf"
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        kobert_package,
    ],
    check=True,
)


# 5. 기존 프로젝트 모듈의 import 경로 등록
for path in [PROJECT_DIR, SRC_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print("환경 설정 완료")

Mounted at /content/drive
환경 설정 완료


In [ ]:
import gc, json, time, hashlib,inspect, platform, importlib.metadata
from contextlib import nullcontext
import numpy as np
import pandas as pd
import joblib, torch, torch.nn as nn
import sentencepiece as spm
from kiwipiepy import Kiwi
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from transformers import BertModel, ElectraModel, AutoTokenizer
from kobert_tokenizer import KoBERTTokenizer
from sklearn.metrics import f1_score, hamming_loss, average_precision_score
from IPython.display import display
import preprocessing.text as text_preprocessing
import utils.labels as utils_labels
BASE = Path('/content/drive/MyDrive/26_KOTE')
DATA_DIR = BASE / 'data'
OUT = BASE / 'artifacts/week12_comparison'
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', '추론 속도 비교를 위해 GPU 런타임을 선택하세요.'
LABEL_NAMES = list(utils_labels.LABELS)
NUM_LABELS = len(LABEL_NAMES)
assert NUM_LABELS == 44

SPECS = {
    'lr': { 'folder': 'week4_lr', 'file': 'logistic_regression.joblib' },
    'svm': { 'folder': 'week5_svm', 'file': 'linear_svm.joblib' },
    'textcnn': { 'folder': 'week8_tuning', 'run_id': 'c660b5e3818a' },
    'bilstm': { 'folder': 'week8_tuning', 'run_id': 'c70d37cbf31c' },
    'kobert': { 'folder': 'week9_kobert', 'run_id': '043c26b05abf' },
    'koelectra': { 'folder': 'week11_koelectra', 'run_id': '495dd8911426' },
}

BATCH_SIZE = 64

def read_json(p):
  return json.loads(Path(p).read_text(encoding='utf-8'))

def save_json(obj, p):
  Path(p).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')

def sync():
  if DEVICE.type == 'cuda':
    torch.cuda.synchronize()

def release():
  gc.collect()
  if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

def sha256(p):
  h = hashlib.sha256()
  with open(p, 'rb') as f:
    for block in iter(lambda: f.read(1024 * 1024), b''):
      h.update(block)
  return h.hexdigest()

ENV = {
    'python': sys.version,
    'platform': platform.platform(),
    'gpu': torch.cuda.get_device_name(0),
    'packages': {k: importlib.metadata.version(k) for k in ['torch', 'transformers', 'scikit-learn', 'numpy', 'kiwipiepy', 'sentencepiece']},
    'preprocessing_source_sha256': sha256(inspect.getfile(text_preprocessing)),
    'labels': LABEL_NAMES,
    'evaluation_batch_size': BATCH_SIZE,
    'transformer_autocast': 'float16',
    'other_neural_precision': 'float32',
    'project_commit': subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], text=True).strip()
}

save_json(ENV, OUT / 'environment.json')
print('GPU:', ENV['gpu'])
display(pd.Series(ENV['packages']))

GPU: NVIDIA L4


,0
torch,2.14.1+cu130
transformers,4.57.6
scikit-learn,1.6.1
numpy,2.4.6
kiwipiepy,0.24.1
sentencepiece,0.2.2


In [ ]:
CONFIGS, SUMMARIES = {}, {}
for name, spec in SPECS.items():
  folder = BASE / 'artifacts' / spec['folder']
  spec['artifact_dir'] = str(folder)
  run  = folder / 'runs' / spec['run_id'] if 'run_id' in spec else folder
  spec['run_dir'] = str(run)
  cfg = read_json(run / 'config.json')
  CONFIGS[name] = cfg

  if 'run_id' in spec:
    assert(run / 'best.pt').is_file(), run / 'best.pt'
    summary = read_json(run / 'summary.json')
    assert summary['run_id'] == spec['run_id'] and summary['config'] == cfg
    expected = read_json(folder / ('final_configs.json' if name in ['textcnn', 'bilstm'] else 'final_config.json'))

    if name in ['textcnn', 'bilstm']:
      expected = expected[name]
    assert cfg == expected, f'{name}: 선택 run과 최종 설정이 다릅니다.'
    SUMMARIES[name] = summary

    if (run / 'label_names.json').exists():
      assert read_json(run / 'label_names.json') == LABEL_NAMES
  else:
    for f in [spec['file'], 'tfidf_vectorizer.joblib', 'multilabel_binarizer.joblib']:
      assert (folder / f).is_file(), folder / f

save_json({ 'specs': SPECS, 'configs': CONFIGS }, OUT / 'model_manifest.json')
display(pd.DataFrame([{
    'model': k,
    'run_id': v.get('run_id', 'joblib'),
    'preprocessing': CONFIGS[k]['preprocessing'],
    'path': v['run_dir']
} for k,v in SPECS.items()]))

,model,run_id,preprocessing,path
0,lr,joblib,repeat,/content/drive/MyDrive/26_KOTE/artifacts/week4_lr
1,svm,joblib,repeat,/content/drive/MyDrive/26_KOTE/artifacts/week5...
2,textcnn,c660b5e3818a,repeat,/content/drive/MyDrive/26_KOTE/artifacts/week8...
3,bilstm,c70d37cbf31c,repeat,/content/drive/MyDrive/26_KOTE/artifacts/week8...
4,kobert,043c26b05abf,repeat,/content/drive/MyDrive/26_KOTE/artifacts/week9...
5,koelectra,495dd8911426,repeat,/content/drive/MyDrive/26_KOTE/artifacts/week1...


In [ ]:
def load_split(split):
    return pd.read_csv(DATA_DIR / f'{split}.tsv', sep='\t', header=None, names=['id','text','labels'])
def make_labels(df):
    y = np.zeros((len(df), NUM_LABELS), dtype=np.int32)
    for i, value in enumerate(df['labels'].astype(str)):
        ids = [int(x.strip()) for x in value.split(',') if x.strip()]
        assert all(0 <= x < NUM_LABELS for x in ids)
        y[i, ids] = 1
    return y

def preprocess(name, texts):
    cfg = CONFIGS[name]
    if name in ['lr','svm']:
        kwargs = {k: cfg[k] for k in ['normalize_repeat','remove_symbols']}
        series = pd.Series(texts).astype(str)
    else:
        kwargs = {'normalize_repeat': cfg['preprocessing'] != 'minimal',
                  'remove_symbols': cfg['preprocessing'] == 'repeat_symbols'}
        series = pd.Series(texts).fillna('').astype(str)
    return [text_preprocessing.preprocess_text(t, **kwargs) for t in series]

def metrics(y, scores, threshold):
    scores = np.asarray(scores)
    assert scores.shape == y.shape and np.isfinite(scores).all()
    pred = scores >= np.asarray(threshold)
    return {'micro_f1': float(f1_score(y,pred,average='micro',zero_division=0)),
            'macro_f1': float(f1_score(y,pred,average='macro',zero_division=0)),
            'hamming_loss': float(hamming_loss(y,pred)),
            'ap_micro': float(average_precision_score(y,scores,average='micro'))}

def threshold_search(y, scores, grid):
    # AP는 임계값과 무관하므로 탐색 반복문 밖에서 계산
    rows = [{'threshold':float(t), 'micro_f1':float(f1_score(y,scores >= t,average='micro',zero_division=0))} for t in grid]
    table = pd.DataFrame(rows)
    return float(table.loc[table.micro_f1.idxmax(),'threshold']), table

val_df = load_split('val'); y_val = make_labels(val_df)
assert y_val.shape == (5000,44) and val_df.id.is_unique
print('Validation:', y_val.shape)


Validation: (5000, 44)


In [ ]:
class SPTokenizer:
    def __init__(self, path):
        self.sp = spm.SentencePieceProcessor(model_file=str(path))
        self.pad_id = self.sp.pad_id()
        self.unk_id = self.sp.unk_id()
        self.vocab_size = self.sp.get_piece_size()

    def encode(self, text):
        return self.sp.encode(str(text), out_type=int)

class KiwiTokenizer:
    def __init__(self, vocab_path):
        self.kiwi = Kiwi()
        self.vocab = json.loads(
            Path(vocab_path).read_text(encoding="utf-8")
        )
        self.pad_id = 0
        self.unk_id = 1
        self.vocab_size = len(self.vocab)

    def pieces(self, text):
        return [
            f"{token.form}/{token.tag}"
            for token in self.kiwi.tokenize(str(text))
        ]

    def encode(self, text):
        return [
            self.vocab.get(piece, self.unk_id)
            for piece in self.pieces(text)
        ]

class TextCNN(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, config["embed_dim"], padding_idx=0
        )
        self.kernel_sizes = config["kernel_sizes"]
        self.mask_padding = config["cnn_mask"]

        self.convs = nn.ModuleList([
            nn.Conv1d(
                config["embed_dim"],
                config["num_filters"],
                kernel_size=k
            )
            for k in self.kernel_sizes
        ])

        self.dropout = nn.Dropout(config["dropout"])
        self.classifier = nn.Linear(
            config["num_filters"] * len(self.kernel_sizes),
            NUM_LABELS
        )

    def forward(self, input_ids, lengths):
        x = self.embedding(input_ids).transpose(1, 2)
        outputs = []

        for conv in self.convs:
            features = torch.relu(conv(x))

            if self.mask_padding:
                starts = torch.arange(
                    features.size(2), device=features.device
                )[None, :]

                valid = starts < lengths.to(features.device)[:, None]
                features = features.masked_fill(
                    ~valid[:, None, :],
                    torch.finfo(features.dtype).min
                )

            outputs.append(features.max(dim=2).values)

        features = torch.cat(outputs, dim=1)
        return self.classifier(self.dropout(features))

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()

        self.pooling = config["pooling"]

        self.embedding = nn.Embedding(
            vocab_size, config["embed_dim"], padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=config["embed_dim"],
            hidden_size=config["hidden_dim"],
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        feature_dim = config["hidden_dim"] * 2
        if self.pooling == "meanmax":
            feature_dim *= 2

        self.dropout = nn.Dropout(config["dropout"])
        self.classifier = nn.Linear(feature_dim, NUM_LABELS)

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, (hidden, _) = self.lstm(packed)

        if self.pooling == "last":
            features = torch.cat(
                [hidden[-2], hidden[-1]], dim=1
            )

        else:
            output, _ = pad_packed_sequence(
                packed_output, batch_first=True
            )

            mask = (
                torch.arange(output.size(1), device=output.device)[None, :]
                < lengths.to(output.device)[:, None]
            )

            maximum = output.masked_fill(
                ~mask[:, :, None],
                torch.finfo(output.dtype).min
            ).max(dim=1).values

            if self.pooling == "max":
                features = maximum

            elif self.pooling == "meanmax":
                mean = (
                    output.masked_fill(~mask[:, :, None], 0).sum(dim=1)
                    / lengths.to(output.device)[:, None]
                )
                features = torch.cat([mean, maximum], dim=1)

            else:
                raise ValueError(self.pooling)

        return self.classifier(self.dropout(features))

class KoBERTClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.bert = BertModel.from_pretrained(
            config["model_name"]
        )
        self.dropout = nn.Dropout(config["dropout"])
        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            NUM_LABELS,
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            # 단일 문장이므로 token_type_ids는 기본값 0 사용
            return_dict=True,
        )

        features = self.dropout(outputs.pooler_output)
        return self.classifier(features)

class KoELECTRAClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.electra = ElectraModel.from_pretrained(
            config["model_name"]
        )
        self.dropout = nn.Dropout(config["dropout"])
        # KoBERT의 pooler에 대응하는 학습 가능한 문장 변환층
        self.dense = nn.Linear(self.electra.config.hidden_size, self.electra.config.hidden_size)
        self.classifier = nn.Linear(
            self.electra.config.hidden_size,
            NUM_LABELS,
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.electra(
            input_ids=input_ids,
            attention_mask=attention_mask,
            # 단일 문장이므로 token_type_ids는 기본값 0 사용
            return_dict=True,
        )

        features = torch.tanh(self.dense(outputs.last_hidden_state[:, 0, :]))
        features = self.dropout(features)
        return self.classifier(features)


In [ ]:
def load_bundle(name):
    spec, cfg = SPECS[name], CONFIGS[name]
    folder, run = Path(spec['artifact_dir']), Path(spec['run_dir'])

    if name in ['lr','svm']:
        model = joblib.load(folder / spec['file'])
        vectorizer = joblib.load(folder / 'tfidf_vectorizer.joblib')
        mlb = joblib.load(folder / 'multilabel_binarizer.joblib')
        assert np.array_equal(mlb.classes_, np.arange(NUM_LABELS)), '레이블 열 순서 불일치'
        return {'model':model, 'tokenizer':vectorizer}

    if name in ['textcnn','bilstm']:
        key = (cfg['preprocessing'], cfg['tokenizer'], cfg['vocab_size'])
        tok_dir = folder / 'tokenizers' / f'{key[0]}_{key[1]}_{key[2]}'

        if key == ('repeat','unigram',8000):
            path = BASE / 'artifacts/week6_textcnn/sentencepiece/kote_sp.model'
            assert path.is_file(), path
            tokenizer = SPTokenizer(path)

        elif cfg['tokenizer'] in ['unigram','bpe']:
            path = tok_dir / 'tokenizer.model'; assert path.is_file(), path
            tokenizer = SPTokenizer(path)

        elif cfg['tokenizer'] == 'kiwi':
            path = tok_dir / 'vocab.json'; assert path.is_file(), path
            tokenizer = KiwiTokenizer(path)

        else: raise ValueError(cfg['tokenizer'])
        model = (TextCNN if name == 'textcnn' else BiLSTM)(tokenizer.vocab_size,cfg)

    elif name == 'kobert':
        tokenizer = KoBERTTokenizer.from_pretrained(cfg['model_name'])
        assert tokenizer.encode('한국어 모델을 공유합니다.') == [2,4958,6855,2046,7088,1050,7843,54,3]
        model = KoBERTClassifier(cfg)

    else:
        tokenizer = AutoTokenizer.from_pretrained(folder / 'tokenizer', use_fast=True)
        model = KoELECTRAClassifier(cfg)

    if name in ['kobert','koelectra']:
        tokenizer.padding_side = 'right'; tokenizer.truncation_side = 'right'
    state = torch.load(run / 'best.pt', map_location='cpu', weights_only=True)
    model.load_state_dict(state, strict=True)
    model.to(DEVICE).eval()
    return {'model':model,'tokenizer':tokenizer}

@torch.inference_mode()
def predict(name, bundle, texts, batch_size=BATCH_SIZE):
    # 전처리·토큰화·CPU/GPU 전송·점수의 CPU 반환까지 포함
    cfg = CONFIGS[name]
    processed = preprocess(name,texts)
    model, tok = bundle['model'], bundle['tokenizer']
    if name in ['lr','svm']:
        x = tok.transform(processed)
        scores = model.predict_proba(x) if name == 'lr' else model.decision_function(x)
        return np.asarray(scores, dtype=np.float32)
    result = []

    for start in range(0,len(processed),batch_size):
        batch = processed[start:start+batch_size]
        if name in ['textcnn','bilstm']:
            ids = [tok.encode(t)[:cfg['max_len']] or [tok.unk_id] for t in batch]
            lengths = torch.tensor([len(x) for x in ids],dtype=torch.long)
            inputs = torch.tensor([x+[0]*(cfg['max_len']-len(x)) for x in ids],dtype=torch.long,device=DEVICE)
            logits = model(inputs,lengths)
        else:
            encoded = tok(batch,padding='max_length',truncation=True,max_length=cfg['max_len'],
                          return_attention_mask=True,return_token_type_ids=False,return_tensors='pt')
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                logits = model(encoded['input_ids'].to(DEVICE),encoded['attention_mask'].to(DEVICE))
        result.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.vstack(result)

def save_predictions(name, split, df, y, scores, threshold=None):
    fields = {'ids':df.id.astype(str).to_numpy(), 'labels':y, 'scores':scores,
              'label_names':np.asarray(LABEL_NAMES,dtype=str)}
    if threshold is not None: fields['predictions'] = (scores >= np.asarray(threshold)).astype(np.uint8)
    np.savez_compressed(OUT / f'{name}_{split}_predictions.npz',**fields)


In [ ]:
VAL_SCORES, reproduction = {}, []

for name in SPECS:
    print('검증 추론:',name)
    bundle = load_bundle(name)
    scores = predict(name, bundle, val_df.text.tolist())
    assert scores.shape == y_val.shape and np.isfinite(scores).all()
    VAL_SCORES[name] = scores
    coarse = CONFIGS[name]['threshold'] if name in ['lr','svm'] else SUMMARIES[name]['threshold']
    row = {'model':name, 'threshold':coarse, **metrics(y_val,scores,coarse)}

    if name in SUMMARIES:
        row['original_micro_f1'] = SUMMARIES[name]['micro_f1']
        row['difference'] = row['micro_f1']-row['original_micro_f1']

    if name in ['lr','svm'] and CONFIGS[name]['preprocessing'] in ['repeat','repeat_symbols']:
        # 업로드한 원본 노트북의 선택 조건 검증값(소수점 6자리)
        row['original_micro_f1'] = {'lr':0.512276,'svm':0.486014}[name]
        row['difference'] = row['micro_f1']-row['original_micro_f1']
    save_predictions(name,'val',val_df,y_val,scores)
    reproduction.append(row)
    del bundle; release()
reproduction = pd.DataFrame(reproduction)
display(reproduction)
reproduction.to_csv(OUT / 'validation_reproduction.csv',index=False)
assert reproduction['difference'].dropna().abs().max() <= 0.001, '검증 재현 오차 확인 후 진행하세요.'


검증 추론: lr
검증 추론: svm
검증 추론: textcnn
검증 추론: bilstm
검증 추론: kobert


tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

... (2 outputs left out)

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/452M [00:00<?, ?B/s]

,model,threshold,micro_f1,macro_f1,hamming_loss,ap_micro,original_micro_f1,difference
0,lr,0.25,0.512276,0.365655,0.200636,0.523327,0.512276,-2.819827e-08
1,svm,-0.40,0.486014,0.387477,0.216914,0.490526,0.486014,3.249502e-07
2,textcnn,0.25,0.569636,0.452469,0.176136,0.602253,0.569636,0.000000e+00
3,bilstm,0.30,0.585562,0.466847,0.156364,0.628908,0.585562,0.000000e+00
4,kobert,0.30,0.618778,0.513957,0.150432,0.668343,0.618727,5.154651e-05
5,koelectra,0.30,0.636463,0.540466,0.142500,0.690601,0.636463,0.000000e+00


In [ ]:
GRID = np.round(np.arange(0.01,1.00,0.01),2)
POLICIES, selection_rows = {}, []
def choose_policy(name,scores):
    common, table = threshold_search(y_val,scores,GRID)
    thresholds = np.full(NUM_LABELS,common,dtype=float)
    for j in range(NUM_LABELS):
        support = int(y_val[:,j].sum())
        if support < 20: continue
        fs = np.array([f1_score(y_val[:,j],scores[:,j] >= t,zero_division=0) for t in GRID])
        ties = np.flatnonzero(np.isclose(fs,fs.max()))
        t = GRID[ties[np.argmin(abs(GRID[ties]-common))]]
        strength = support/(support+100)
        thresholds[j] = strength*t+(1-strength)*common
    options = [('global',common),('labelwise_shrunk',thresholds.tolist())]
    rows = [{'model':name,'method':method,**metrics(y_val,scores,t)} for method,t in options]
    winner = max(range(len(rows)),key=lambda i: rows[i]['micro_f1'])
    table.to_csv(OUT / f'{name}_global_threshold_search.csv',index=False)
    POLICIES[name] = {'method':options[winner][0], 'threshold':options[winner][1]}
    for i,row in enumerate(rows): row['selected'] = i == winner
    selection_rows.extend(rows)

for name,scores in VAL_SCORES.items():
    if name in ['lr','svm']:
        t = float(CONFIGS[name]['threshold'])
        POLICIES[name] = {'method':'original_global','threshold':t}
        selection_rows.append({'model':name,'method':'original_global','selected':True,**metrics(y_val,scores,t)})
    else: choose_policy(name,scores)

ensemble_rows = []
for alpha in [0.25,0.5,0.75]:
    name = f'ensemble_kobert_{alpha:.2f}'
    scores = alpha*VAL_SCORES['kobert']+(1-alpha)*VAL_SCORES['koelectra']
    choose_policy(name,scores)
    ensemble_rows.append({'model':name,'alpha_kobert':alpha,**metrics(y_val,scores,POLICIES[name]['threshold'])})
ensemble_table = pd.DataFrame(ensemble_rows)
winner = ensemble_table.loc[ensemble_table.micro_f1.idxmax()]
ENSEMBLE_NAME = str(winner['model']); ALPHA = float(winner['alpha_kobert'])
VAL_SCORES[ENSEMBLE_NAME] = ALPHA*VAL_SCORES['kobert']+(1-ALPHA)*VAL_SCORES['koelectra']
selection_table = pd.DataFrame(selection_rows)
display(selection_table); display(ensemble_table)
selection_table.to_csv(OUT / 'validation_threshold_comparison.csv',index=False)
ensemble_table.to_csv(OUT / 'validation_ensemble_comparison.csv',index=False)
print('선택한 앙상블:',ENSEMBLE_NAME)


,model,method,selected,micro_f1,macro_f1,hamming_loss,ap_micro
0,lr,original_global,True,0.512276,0.365655,0.200636,0.523327
1,svm,original_global,True,0.486014,0.387477,0.216914,0.490526
2,textcnn,global,True,0.569813,0.447528,0.172241,0.602253
3,textcnn,labelwise_shrunk,False,0.563996,0.476499,0.185423,0.602253
4,bilstm,global,True,0.585871,0.469933,0.158755,0.628908
5,bilstm,labelwise_shrunk,False,0.576793,0.493111,0.173059,0.628908
6,kobert,global,True,0.619245,0.512362,0.148368,0.668343
7,kobert,labelwise_shrunk,False,0.613802,0.529696,0.155805,0.668343
8,koelectra,global,True,0.637282,0.536037,0.137764,0.690601
9,koelectra,labelwise_shrunk,False,0.632215,0.550498,0.147418,0.690601


,model,alpha_kobert,micro_f1,macro_f1,hamming_loss,ap_micro
0,ensemble_kobert_0.25,0.25,0.641739,0.536647,0.135923,0.700609
1,ensemble_kobert_0.50,0.50,0.640954,0.531955,0.136718,0.700297
2,ensemble_kobert_0.75,0.75,0.632339,0.523431,0.141668,0.690497


선택한 앙상블: ensemble_kobert_0.25


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import f1_score


ANALYSIS_DIR = OUT / "ensemble_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

truth = np.asarray(y_val, dtype=bool)

bert_scores = np.asarray(VAL_SCORES["kobert"])
electra_scores = np.asarray(VAL_SCORES["koelectra"])

ensemble_scores = (
    ALPHA * bert_scores
    + (1 - ALPHA) * electra_scores
)

assert bert_scores.shape == truth.shape
assert electra_scores.shape == truth.shape
assert len(val_df) == len(truth)

score_map = {
    "kobert": bert_scores,
    "koelectra": electra_scores,
    ENSEMBLE_NAME: ensemble_scores,
}

print(
    f"분석할 앙상블: "
    f"KoBERT {ALPHA:.0%} + KoELECTRA {1 - ALPHA:.0%}"
)
print("분석 데이터: 검증셋")
print("기존 모델·임계값·앙상블 선택은 변경하지 않습니다.")


# --------------------------------------------------
# 1. 공통 임계값 방식과 최종 선택 방식의 성능 비교
# --------------------------------------------------

performance_rows = []
global_thresholds = {}

for name, scores in score_map.items():
    common_threshold, _ = threshold_search(
        y_val,
        scores,
        GRID,
    )

    global_thresholds[name] = common_threshold

    performance_rows.append({
        "model": name,
        "comparison": "global_only",
        "threshold_method": "global",
        "global_threshold": common_threshold,
        **metrics(y_val, scores, common_threshold),
    })

    policy = POLICIES[name]

    performance_rows.append({
        "model": name,
        "comparison": "selected_policy",
        "threshold_method": policy["method"],
        "global_threshold": (
            float(policy["threshold"])
            if np.isscalar(policy["threshold"])
            else np.nan
        ),
        **metrics(
            y_val,
            scores,
            policy["threshold"],
        ),
    })

performance_table = pd.DataFrame(performance_rows)

print("\n1. 공통 임계값 방식 / 최종 선택 방식")
display(performance_table.round(6))

performance_table.to_csv(
    ANALYSIS_DIR / "performance_by_threshold_method.csv",
    index=False,
    encoding="utf-8-sig",
)


# --------------------------------------------------
# 2. 최종 선택 임계값으로 예측값 생성
# --------------------------------------------------

def threshold_vector(name):
    threshold = np.asarray(
        POLICIES[name]["threshold"],
        dtype=float,
    )

    return np.broadcast_to(
        threshold,
        (truth.shape[1],),
    ).copy()


thresholds = {
    name: threshold_vector(name)
    for name in score_map
}

predictions = {
    name: scores >= thresholds[name]
    for name, scores in score_map.items()
}

bert_pred = predictions["kobert"]
electra_pred = predictions["koelectra"]
ensemble_pred = predictions[ENSEMBLE_NAME]

bert_correct = bert_pred == truth
electra_correct = electra_pred == truth

pair_masks = {
    "둘 다 정답": bert_correct & electra_correct,
    "KoBERT만 정답": bert_correct & ~electra_correct,
    "KoELECTRA만 정답": ~bert_correct & electra_correct,
    "둘 다 오답": ~bert_correct & ~electra_correct,
}


# --------------------------------------------------
# 3. 감정 단위 오류 겹침
# 한 관측 = 문장 하나 × 감정 하나
# 정답 양성/음성을 분리해서 TN의 영향을 확인
# --------------------------------------------------

overlap_rows = []

for scope, scope_mask in [
    ("전체 문장-감정 쌍", np.ones_like(truth)),
    ("정답: 감정 있음", truth),
    ("정답: 감정 없음", ~truth),
]:
    denominator = int(scope_mask.sum())

    for category, category_mask in pair_masks.items():
        count = int(
            (scope_mask & category_mask).sum()
        )

        overlap_rows.append({
            "scope": scope,
            "category": category,
            "count": count,
            "percent_within_scope": (
                100 * count / denominator
                if denominator
                else np.nan
            ),
        })

overlap_table = pd.DataFrame(overlap_rows)

print("\n2. 감정 단위: 두 모델의 정답·오답 조합")
display(overlap_table.round(3))

overlap_table.to_csv(
    ANALYSIS_DIR / "label_decision_overlap.csv",
    index=False,
    encoding="utf-8-sig",
)


# --------------------------------------------------
# 4. 문장 전체를 완벽하게 맞힌 경우 비교
# 감정 목록 전체가 같아야 정답으로 인정
# --------------------------------------------------

bert_exact = bert_correct.all(axis=1)
electra_exact = electra_correct.all(axis=1)

sentence_masks = {
    "둘 다 정답": bert_exact & electra_exact,
    "KoBERT만 정답": bert_exact & ~electra_exact,
    "KoELECTRA만 정답": ~bert_exact & electra_exact,
    "둘 다 오답": ~bert_exact & ~electra_exact,
}

sentence_overlap = pd.DataFrame([
    {
        "category": category,
        "sentences": int(mask.sum()),
        "percent": 100 * float(mask.mean()),
    }
    for category, mask in sentence_masks.items()
])

print("\n3. 문장 단위: 감정 목록 전체를 맞혔는가?")
display(sentence_overlap.round(3))

sentence_overlap.to_csv(
    ANALYSIS_DIR / "sentence_exact_match_overlap.csv",
    index=False,
    encoding="utf-8-sig",
)


# --------------------------------------------------
# 5. 앙상블이 고친 오류와 새로 만든 오류
# 단독 모델 각각을 기준으로 비교
# --------------------------------------------------

def transition_masks(base_pred, combined_pred):
    return {
        "FN→TP: 놓친 감정 발견": (
            truth & ~base_pred & combined_pred
        ),
        "FP→TN: 잘못 예측한 감정 제거": (
            ~truth & base_pred & ~combined_pred
        ),
        "TP→FN: 맞혔던 감정을 놓침": (
            truth & base_pred & ~combined_pred
        ),
        "TN→FP: 없던 감정을 잘못 예측": (
            ~truth & ~base_pred & combined_pred
        ),
    }


transition_rows = []
transition_map = {}

for base_name in ["kobert", "koelectra"]:
    masks = transition_masks(
        predictions[base_name],
        ensemble_pred,
    )

    transition_map[base_name] = masks

    for change, mask in masks.items():
        transition_rows.append({
            "baseline": base_name,
            "change": change,
            "decisions": int(mask.sum()),
            "affected_sentences": int(
                mask.any(axis=1).sum()
            ),
        })

transition_table = pd.DataFrame(transition_rows)

print("\n4. 단독 모델 대비 앙상블의 개선·악화")
display(transition_table)

transition_table.to_csv(
    ANALYSIS_DIR / "ensemble_decision_changes.csv",
    index=False,
    encoding="utf-8-sig",
)


# TP·FP·FN으로 Micro F1 변화를 직접 확인
def confusion_summary(pred):
    tp = int((truth & pred).sum())
    fp = int((~truth & pred).sum())
    fn = int((truth & ~pred).sum())

    denominator = 2 * tp + fp + fn

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "micro_f1": (
            2 * tp / denominator
            if denominator
            else 0.0
        ),
        "exact_match_ratio": float(
            (pred == truth).all(axis=1).mean()
        ),
    }


confusion_table = pd.DataFrame([
    {
        "model": name,
        **confusion_summary(pred),
    }
    for name, pred in predictions.items()
])

print("\n5. TP·FP·FN과 Micro F1")
display(confusion_table.round(6))

confusion_table.to_csv(
    ANALYSIS_DIR / "confusion_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


# --------------------------------------------------
# 6. 실제 사례 출력
# 카테고리별로 서로 다른 문장 최대 5개를 선택
# 점수로 순위를 매기지 않고 고정 seed로 표본 추출
# --------------------------------------------------

N_EXAMPLES = 5
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)


def emotion_list(binary_row):
    indices = np.flatnonzero(binary_row)

    return ", ".join(
        LABEL_NAMES[index]
        for index in indices
    ) or "(예측 없음)"


def sample_cases(mask, category):
    candidate_sentences = np.flatnonzero(
        mask.any(axis=1)
    )

    if not len(candidate_sentences):
        return []

    selected_sentences = rng.choice(
        candidate_sentences,
        size=min(
            N_EXAMPLES,
            len(candidate_sentences),
        ),
        replace=False,
    )

    rows = []

    for sample_index in selected_sentences:
        label_candidates = np.flatnonzero(
            mask[sample_index]
        )

        label_index = int(
            rng.choice(label_candidates)
        )

        sample = val_df.iloc[sample_index]

        rows.append({
            "category": category,
            "sample_id": str(sample["id"]),
            "text": sample["text"],
            "emotion": LABEL_NAMES[label_index],
            "target": int(
                truth[sample_index, label_index]
            ),
            "kobert_score": float(
                bert_scores[sample_index, label_index]
            ),
            "kobert_threshold": float(
                thresholds["kobert"][label_index]
            ),
            "kobert_prediction": int(
                bert_pred[sample_index, label_index]
            ),
            "koelectra_score": float(
                electra_scores[
                    sample_index,
                    label_index,
                ]
            ),
            "koelectra_threshold": float(
                thresholds["koelectra"][label_index]
            ),
            "koelectra_prediction": int(
                electra_pred[
                    sample_index,
                    label_index,
                ]
            ),
            "ensemble_score": float(
                ensemble_scores[
                    sample_index,
                    label_index,
                ]
            ),
            "ensemble_threshold": float(
                thresholds[ENSEMBLE_NAME][label_index]
            ),
            "ensemble_prediction": int(
                ensemble_pred[
                    sample_index,
                    label_index,
                ]
            ),
            "all_true_emotions": emotion_list(
                truth[sample_index]
            ),
            "all_kobert_predictions": emotion_list(
                bert_pred[sample_index]
            ),
            "all_koelectra_predictions": emotion_list(
                electra_pred[sample_index]
            ),
            "all_ensemble_predictions": emotion_list(
                ensemble_pred[sample_index]
            ),
        })

    return rows


example_rows = []

# 서로 다른 판단을 한 사례
for category in [
    "KoBERT만 정답",
    "KoELECTRA만 정답",
]:
    example_rows.extend(
        sample_cases(
            pair_masks[category],
            category,
        )
    )

# 두 단독 모델 중 검증 Micro F1이 더 높은 모델을 기준으로
# 개선·악화 사례 확인. 기존 최종 선택은 변경하지 않음.
best_single = max(
    ["kobert", "koelectra"],
    key=lambda name: metrics(
        y_val,
        score_map[name],
        POLICIES[name]["threshold"],
    )["micro_f1"],
)

for change, mask in transition_map[
    best_single
].items():
    example_rows.extend(
        sample_cases(
            mask,
            f"{best_single} 대비 {change}",
        )
    )

examples = pd.DataFrame(example_rows)

print(
    "\n6. 실제 사례 "
    f"(개선·악화 비교 기준: {best_single})"
)

if examples.empty:
    print("출력할 사례가 없습니다.")
else:
    for category, group in examples.groupby(
        "category",
        sort=False,
    ):
        print(f"\n[{category}]")

        # 세로로 출력해 긴 문장과 점수를 함께 확인
        with pd.option_context(
            "display.max_colwidth",
            160,
            "display.max_rows",
            None,
        ):
            display(
                group.set_index("sample_id")
                .T
                .round(4)
            )

    examples.to_csv(
        ANALYSIS_DIR / "sample_cases.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("\n분석 결과 저장:", ANALYSIS_DIR)

분석할 앙상블: KoBERT 25% + KoELECTRA 75%
분석 데이터: 검증셋
기존 모델·임계값·앙상블 선택은 변경하지 않습니다.

1. 공통 임계값 방식 / 최종 선택 방식


,model,comparison,threshold_method,global_threshold,micro_f1,macro_f1,hamming_loss,ap_micro
0,kobert,global_only,global,0.31,0.619245,0.512362,0.148368,0.668343
1,kobert,selected_policy,global,0.31,0.619245,0.512362,0.148368,0.668343
2,koelectra,global_only,global,0.33,0.637282,0.536037,0.137764,0.690601
3,koelectra,selected_policy,global,0.33,0.637282,0.536037,0.137764,0.690601
4,ensemble_kobert_0.25,global_only,global,0.33,0.641739,0.536647,0.135923,0.700609
5,ensemble_kobert_0.25,selected_policy,global,0.33,0.641739,0.536647,0.135923,0.700609



2. 감정 단위: 두 모델의 정답·오답 조합


,scope,category,count,percent_within_scope
0,전체 문장-감정 쌍,둘 다 정답,179024,81.375
1,전체 문장-감정 쌍,KoBERT만 정답,8335,3.789
2,전체 문장-감정 쌍,KoELECTRA만 정답,10668,4.849
3,전체 문장-감정 쌍,둘 다 오답,21973,9.988
4,정답: 감정 있음,둘 다 정답,23555,59.845
5,정답: 감정 있음,KoBERT만 정답,2988,7.591
6,정답: 감정 있음,KoELECTRA만 정답,3070,7.800
7,정답: 감정 있음,둘 다 오답,9747,24.764
8,정답: 감정 없음,둘 다 정답,155469,86.066
9,정답: 감정 없음,KoBERT만 정답,5347,2.960



3. 문장 단위: 감정 목록 전체를 맞혔는가?


,category,sentences,percent
0,둘 다 정답,17,0.34
1,KoBERT만 정답,20,0.40
2,KoELECTRA만 정답,34,0.68
3,둘 다 오답,4929,98.58


... (13 outputs left out)

sample_id,2993,26581,16971,24415,8037
category,koelectra 대비 TP→FN: 맞혔던 감정을 놓침,koelectra 대비 TP→FN: 맞혔던 감정을 놓침,koelectra 대비 TP→FN: 맞혔던 감정을 놓침,koelectra 대비 TP→FN: 맞혔던 감정을 놓침,koelectra 대비 TP→FN: 맞혔던 감정을 놓침
text,한번만 더 이런영화 만들면 그땐 쳐들어간다.. 나라망신은 다시키고있어..,그냥.. 다음 대통령이다 뭔 말이 필요해?,40덬 잘생겼다기보다는 적당히 센스 있고,디자인도 예쁘고 베란다 밖에 설치하였더니 수신율 70~80% 로 잘나오네요~ 실외에 설치해놔서 내구성도 좋으면 따봉이겠네요,벽쪽으로 바로연결하지말고 허공으로 조립해서 위에서 물부어보세요찔찔찔나오면 벽배관문제가 아닌거죠
emotion,비장함,깨달음,우쭐댐/무시함,아껴주는,우쭐댐/무시함
target,1,1,1,1,1
kobert_score,0.018942,0.046987,0.309858,0.144752,0.161056
kobert_threshold,0.31,0.31,0.31,0.31,0.31
kobert_prediction,0,0,0,0,0
koelectra_score,0.351534,0.358241,0.332527,0.370339,0.377541
koelectra_threshold,0.33,0.33,0.33,0.33,0.33
koelectra_prediction,1,1,1,1,1



[koelectra 대비 TN→FP: 없던 감정을 잘못 예측]


sample_id,18592,317,4826,11436,39208
category,koelectra 대비 TN→FP: 없던 감정을 잘못 예측,koelectra 대비 TN→FP: 없던 감정을 잘못 예측,koelectra 대비 TN→FP: 없던 감정을 잘못 예측,koelectra 대비 TN→FP: 없던 감정을 잘못 예측,koelectra 대비 TN→FP: 없던 감정을 잘못 예측
text,토비 맥과이어 나오는건 당연한거 아니었나 지금 거의 울버린 배우급 인지도이자나,이건 아니지. 우리 장병들이 나라지키다가 적에게 죽었는데 음모설이라니... 감독이 제정신이 아닌거 같다.,히스레저..그의 연기가 너무나도 그립다 조커라는 캐릭터도 사랑스럽다.,본문보니깐 원래 있던 취미도 여자가 정리시킨 느낌이 솔솔나는데 이거 마지막으로 정리시키면 결혼관계도 정리하고 끝날듯,썸네일에 어울리는 제목을 뽑는다면 교도소에서 개과천선하여 요리사가 된 사람이 있다?
emotion,감동/감탄,증오/혐오,기쁨,의심/불신,우쭐댐/무시함
target,0,0,0,0,0
kobert_score,0.960137,0.485523,0.656218,0.46308,0.541288
kobert_threshold,0.31,0.31,0.31,0.31,0.31
kobert_prediction,1,1,1,1,1
koelectra_score,0.165863,0.316898,0.274449,0.307358,0.300746
koelectra_threshold,0.33,0.33,0.33,0.33,0.33
koelectra_prediction,0,0,0,0,0



분석 결과 저장: /content/drive/MyDrive/26_KOTE/artifacts/week12_comparison/ensemble_analysis


In [ ]:
FINAL_NAMES = list(SPECS)+[ENSEMBLE_NAME]
selection = {'models':{k:POLICIES[k] for k in FINAL_NAMES}, 'ensemble_name':ENSEMBLE_NAME,
             'alpha_kobert':ALPHA,'specs':SPECS,'configs':CONFIGS,
             'data_sha256':{s:sha256(DATA_DIR / f'{s}.tsv') for s in ['val','test']},
             'preprocessing_source_sha256':ENV['preprocessing_source_sha256'],'labels':LABEL_NAMES}
lock_path = OUT / 'frozen_selection.json'
if lock_path.exists():
    assert read_json(lock_path) == selection, '이미 고정한 선택과 다릅니다. 테스트 결과를 보고 선택을 변경하지 마세요.'
else: save_json(selection,lock_path)
# 체크포인트 파일의 실제 존재/strict 로드 검증은 앞 검증 추론에서 완료됨
for name in FINAL_NAMES:
    save_predictions(name,'val',val_df,y_val,VAL_SCORES[name],POLICIES[name]['threshold'])
test_df = load_split('test'); y_test = make_labels(test_df)
assert y_test.shape == (5000,NUM_LABELS) and test_df.id.is_unique
assert set(val_df.id).isdisjoint(set(test_df.id)), '검증/테스트 ID 중복'
TEST_SCORES, test_rows = {}, []
for name in SPECS:
    print('테스트 추론:',name)
    bundle = load_bundle(name)
    scores = predict(name,bundle,test_df.text.tolist())
    TEST_SCORES[name] = scores
    t = POLICIES[name]['threshold']
    test_rows.append({'model':name,'threshold_method':POLICIES[name]['method'],**metrics(y_test,scores,t)})
    save_predictions(name,'test',test_df,y_test,scores,t)
    del bundle; release()
scores = ALPHA*TEST_SCORES['kobert']+(1-ALPHA)*TEST_SCORES['koelectra']
TEST_SCORES[ENSEMBLE_NAME] = scores
t = POLICIES[ENSEMBLE_NAME]['threshold']
test_rows.append({'model':ENSEMBLE_NAME,'threshold_method':POLICIES[ENSEMBLE_NAME]['method'],**metrics(y_test,scores,t)})
save_predictions(ENSEMBLE_NAME,'test',test_df,y_test,scores,t)
test_table = pd.DataFrame(test_rows).sort_values('micro_f1',ascending=False)
test_table.to_csv(OUT / 'test_metrics.csv',index=False)
display(test_table)
# 다음 주 원문·정답·모델 예측을 ID로 연결할 수 있도록 보존
val_df.to_csv(OUT / 'validation_samples.tsv',sep='\t',index=False)
test_df.to_csv(OUT / 'test_samples.tsv',sep='\t',index=False)


테스트 추론: lr
테스트 추론: svm
테스트 추론: textcnn
테스트 추론: bilstm
테스트 추론: kobert


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


테스트 추론: koelectra


,model,threshold_method,micro_f1,macro_f1,hamming_loss,ap_micro
6,ensemble_kobert_0.25,global,0.637075,0.530911,0.137955,0.697456
5,koelectra,global,0.630519,0.528922,0.140436,0.687325
4,kobert,global,0.618132,0.510164,0.148823,0.666337
3,bilstm,global,0.582669,0.465671,0.159741,0.628373
2,textcnn,global,0.565435,0.441041,0.174586,0.598028
0,lr,original_global,0.510730,0.361891,0.201264,0.519464
1,svm,original_global,0.481016,0.382939,0.218709,0.485481


In [ ]:
BENCH_N = 1000
BENCH_REPEATS = 3
bench_texts = val_df.text.iloc[:BENCH_N].tolist()
single_texts = bench_texts[:50]
speed_rows, speed_raw = [], []
for name in FINAL_NAMES:
    print('속도 측정:',name)
    if name == ENSEMBLE_NAME:
        b1 = load_bundle('kobert'); b2 = load_bundle('koelectra')
        def call(texts,batch_size):
            return ALPHA*predict('kobert',b1,texts,batch_size)+(1-ALPHA)*predict('koelectra',b2,texts,batch_size)
    else:
        bundle = load_bundle(name)
        def call(texts,batch_size): return predict(name,bundle,texts,batch_size)
    call(bench_texts[:64],BATCH_SIZE); call(single_texts[:1],1); sync()
    batch_times, single_times = [], []
    for repeat in range(BENCH_REPEATS):
        sync(); start = time.perf_counter()
        call(bench_texts,BATCH_SIZE); sync()
        batch_s = time.perf_counter()-start
        sync(); start = time.perf_counter()
        for text in single_texts: call([text],1)
        sync(); single_s = time.perf_counter()-start
        batch_times.append(batch_s); single_times.append(single_s/len(single_texts))
        speed_raw.append({'model':name,'repeat':repeat+1,'batch_seconds':batch_s,
                          'single_mean_seconds':single_times[-1], 'batch_sentences':BENCH_N})
    mean_batch = float(np.mean(batch_times))
    speed_rows.append({'model':name,'device':'cpu' if name in ['lr','svm'] else ENV['gpu'],
                       'batch_seconds_mean':mean_batch,'batch_seconds_std':float(np.std(batch_times,ddof=1)),
                       'sentences_per_second':BENCH_N/mean_batch,
                       'single_latency_ms_mean':float(np.mean(single_times)*1000),
                       'single_latency_ms_std':float(np.std(single_times,ddof=1)*1000)})
    del call
    if name == ENSEMBLE_NAME: del b1,b2
    else: del bundle
    release()
    pd.DataFrame(speed_rows).to_csv(OUT / 'inference_speed.csv',index=False)
    pd.DataFrame(speed_raw).to_csv(OUT / 'inference_speed_raw.csv',index=False)
speed_table = pd.DataFrame(speed_rows); display(speed_table)


속도 측정: lr
속도 측정: svm
속도 측정: textcnn
속도 측정: bilstm
속도 측정: kobert


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


속도 측정: koelectra
속도 측정: ensemble_kobert_0.25


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


,model,device,batch_seconds_mean,batch_seconds_std,sentences_per_second,single_latency_ms_mean,single_latency_ms_std
0,lr,cpu,0.038221,0.000753,26163.556818,6.870341,0.117268
1,svm,cpu,0.036551,0.000237,27359.005888,5.356855,0.009586
2,textcnn,NVIDIA L4,1.294573,0.002773,772.455665,2.430460,0.082704
3,bilstm,NVIDIA L4,1.316525,0.006993,759.575383,3.626724,0.051818
4,kobert,NVIDIA L4,1.390515,0.001451,719.158006,10.966954,0.040751
5,koelectra,NVIDIA L4,1.106773,0.000896,903.527352,14.005973,0.271225
6,ensemble_kobert_0.25,NVIDIA L4,2.498451,0.005180,400.247948,25.324446,0.571951


In [ ]:
CLASSICAL_TRAINING = {
    'lr': {'seconds':None, 'device':'CPU (상세 미기록)'},
    'svm': {'seconds':None, 'device':'CPU (상세 미기록)'},
}
# 현재 프로젝트에 저장된 원본 비교표가 있으면 선택한 전처리의 시간을 우선 사용.
# 파일이 없으면 원본 노트북 출력값을 출처와 함께 사용한다. LR의 다른 노트북에는
# 재실행으로 보이는 다른 시간이 있어, LR 원본 파일의 출력만 사용한다.
NOTEBOOK_TIMES = {
    'lr': {'repeat':83.445783,'repeat_symbols':83.782313,'raw':78.407964},
    'svm': {'repeat':15.332739,'repeat_symbols':13.583908,'raw':14.903393},
}
for name, filename in [('lr','week04_lr_preprocessing_comparison.csv'),('svm','week05_svm_preprocessing_comparison.csv')]:
    mode = CONFIGS[name]['preprocessing']
    if CLASSICAL_TRAINING[name]['seconds'] is None:
        path = PROJECT_DIR / 'results' / filename
        if path.exists():
            table = pd.read_csv(path)
            match = table.loc[table.preprocessing == mode]
            if not match.empty:
                CLASSICAL_TRAINING[name]['seconds'] = float(match.iloc[0]['train_time'])
                CLASSICAL_TRAINING[name]['source'] = str(path)
        if CLASSICAL_TRAINING[name]['seconds'] is None:
            CLASSICAL_TRAINING[name]['seconds'] = NOTEBOOK_TIMES[name].get(mode)
            CLASSICAL_TRAINING[name]['source'] = '업로드 원본 노트북 출력 (실제 파일의 학습 실행과 동일한지 확인 필요)'
save_json(CLASSICAL_TRAINING, OUT / 'classical_training_sources.json')
training_rows = []
for name in SPECS:
    if name in SUMMARIES:
        s = SUMMARIES[name]
        seconds,device = s['training_seconds'],s.get('gpu') or s.get('device','미기록')
    else: seconds,device = CLASSICAL_TRAINING[name]['seconds'],CLASSICAL_TRAINING[name]['device']
    training_rows.append({'model':name,'training_seconds':seconds,'training_device':device,
                          'training_scope':'선택 run의 전체 학습; 전처리 제외'})
training_rows.append({'model':ENSEMBLE_NAME,
    'training_seconds':SUMMARIES['kobert']['training_seconds']+SUMMARIES['koelectra']['training_seconds'],
    'training_device':'기반 모델별 기록 참조','training_scope':'두 기반 모델 합계; 추가 학습 없음'})
training_table = pd.DataFrame(training_rows)
training_table.to_csv(OUT / 'training_time.csv',index=False)
final_table = test_table.merge(speed_table,on='model').merge(training_table,on='model')
final_table.to_csv(OUT / 'final_comparison.csv',index=False)
display(final_table)
print('결과 폴더:',OUT)
print('다음 주 분석용: *_test_predictions.npz, test_samples.tsv, frozen_selection.json')


,model,threshold_method,micro_f1,macro_f1,hamming_loss,ap_micro,device,batch_seconds_mean,batch_seconds_std,sentences_per_second,single_latency_ms_mean,single_latency_ms_std,training_seconds,training_device,training_scope
0,ensemble_kobert_0.25,global,0.637075,0.530911,0.137955,0.697456,NVIDIA L4,2.498451,0.005180,400.247948,25.324446,0.571951,2349.555646,기반 모델별 기록 참조,두 기반 모델 합계; 추가 학습 없음
1,koelectra,global,0.630519,0.528922,0.140436,0.687325,NVIDIA L4,1.106773,0.000896,903.527352,14.005973,0.271225,1293.198639,NVIDIA A100-SXM4-40GB,선택 run의 전체 학습; 전처리 제외
2,kobert,global,0.618132,0.510164,0.148823,0.666337,NVIDIA L4,1.390515,0.001451,719.158006,10.966954,0.040751,1056.357007,NVIDIA A100-SXM4-80GB,선택 run의 전체 학습; 전처리 제외
3,bilstm,global,0.582669,0.465671,0.159741,0.628373,NVIDIA L4,1.316525,0.006993,759.575383,3.626724,0.051818,93.280569,NVIDIA L4,선택 run의 전체 학습; 전처리 제외
4,textcnn,global,0.565435,0.441041,0.174586,0.598028,NVIDIA L4,1.294573,0.002773,772.455665,2.430460,0.082704,63.887085,NVIDIA L4,선택 run의 전체 학습; 전처리 제외
5,lr,original_global,0.510730,0.361891,0.201264,0.519464,cpu,0.038221,0.000753,26163.556818,6.870341,0.117268,85.492888,CPU (상세 미기록),선택 run의 전체 학습; 전처리 제외
6,svm,original_global,0.481016,0.382939,0.218709,0.485481,cpu,0.036551,0.000237,27359.005888,5.356855,0.009586,15.332739,CPU (상세 미기록),선택 run의 전체 학습; 전처리 제외


결과 폴더: /content/drive/MyDrive/26_KOTE/artifacts/week12_comparison
다음 주 분석용: *_test_predictions.npz, test_samples.tsv, frozen_selection.json
